In [3]:
import json
import pandas as pd
import os
from tqdm import tqdm

In [6]:
test_df = pd.DataFrame(json.load(open('/home/work/hocheol_dir/workspace/data/034_cropped_data/0512_034_dataset/test.json')))
tr_df = pd.DataFrame(json.load(open('/home/work/hocheol_dir/workspace/data/034_cropped_data/0512_034_dataset/train.json')))
val_df = pd.DataFrame(json.load(open('/home/work/hocheol_dir/workspace/data/034_cropped_data/0512_034_dataset/val.json')))

In [7]:
test_df.head()

,id,left_cheek_path,right_cheek_path,label
0,0001_19,Test/0001/left_cheek_0001_CRS_19_01.jpg,Test/0001/right_cheek_0001_CRS_19_01.jpg,0
1,0001_46,Test/0001/left_cheek_0001_CRS_46_01.jpg,Test/0001/right_cheek_0001_CRS_46_01.jpg,0
2,0006_19,Test/0006/left_cheek_0006_CRS_19_01.jpg,Test/0006/right_cheek_0006_CRS_19_01.jpg,1
3,0006_46,Test/0006/left_cheek_0006_CRS_46_01.jpg,Test/0006/right_cheek_0006_CRS_46_01.jpg,1
4,0009_19,Test/0009/left_cheek_0009_CRS_19_01.jpg,Test/0009/right_cheek_0009_CRS_19_01.jpg,2


In [10]:
sum([len(test_df), len(tr_df), len(val_df)])

5309

In [25]:
len(test_df), len(tr_df), len(val_df)

(532, 4245, 532)

In [13]:
forehead_path = []

for root, dirs, files in tqdm(os.walk('/home/work/hocheol_dir/workspace/data/034_cropped_data/034_forehead')):
    for file in files:
        if file.lower().endswith('.jpg'):
            forehead_path.append(os.path.join(root, file))
forehead_path[0]

2705it [00:01, 1919.13it/s]


'/home/work/hocheol_dir/workspace/data/034_cropped_data/034_forehead/1.Training/원천데이터/1058/forehead_1058_CRS_19_01.jpg'

In [56]:
forehead_data = []

for path in forehead_path:
    userId = path.split('/')[-2]
    if path.lower().endswith('19_01.jpg'):
        id = f'{userId}_19'
    else:
        id = f'{userId}_46'
    forehead_data.append({
        'id': id,
        'origin_forehead_path': path
    })
    

In [57]:
for data in forehead_data:
    if not os.path.exists(data['origin_forehead_path']):
        print(f"File not found: {data['origin_forehead_path']}")

In [58]:
forehead_data_df = pd.DataFrame(forehead_data)
forehead_data_df.head().values

array([['1058_19',
        '/home/work/hocheol_dir/workspace/data/034_cropped_data/034_forehead/1.Training/원천데이터/1058/forehead_1058_CRS_19_01.jpg'],
       ['1058_46',
        '/home/work/hocheol_dir/workspace/data/034_cropped_data/034_forehead/1.Training/원천데이터/1058/forehead_1058_CRS_46_01.jpg'],
       ['1122_46',
        '/home/work/hocheol_dir/workspace/data/034_cropped_data/034_forehead/1.Training/원천데이터/1122/forehead_1122_CRS_46_01.jpg'],
       ['1122_19',
        '/home/work/hocheol_dir/workspace/data/034_cropped_data/034_forehead/1.Training/원천데이터/1122/forehead_1122_CRS_19_01.jpg'],
       ['1155_46',
        '/home/work/hocheol_dir/workspace/data/034_cropped_data/034_forehead/1.Training/원천데이터/1155/forehead_1155_CRS_46_01.jpg']],
      dtype=object)

In [59]:
forehead_data_df[forehead_data_df['id'] == '1443_46']

,id,origin_forehead_path
468,1443_46,/home/work/hocheol_dir/workspace/data/034_crop...


In [60]:
new_test_df = pd.merge(test_df, forehead_data_df, on='id', how='left')
new_tr_df = pd.merge(tr_df, forehead_data_df, on='id', how='left')
new_val_df = pd.merge(val_df, forehead_data_df, on='id', how='left')

In [61]:
len(new_test_df), len(new_tr_df), len(new_val_df)

(532, 4245, 532)

In [62]:
len(test_df), len(tr_df), len(val_df)

(532, 4245, 532)

In [93]:
def make_forehead_path(row):
    dirname = os.path.dirname(row['left_cheek_path'])
    forehead_path = os.path.join(dirname, row['origin_forehead_path'].replace('.JPG', '.jpg').split('/')[-1])
    return forehead_path

In [94]:
new_test_df['forehead_path'] = new_test_df.apply(make_forehead_path, axis=1)
new_tr_df['forehead_path'] = new_tr_df.apply(make_forehead_path, axis=1)
new_val_df['forehead_path'] = new_val_df.apply(make_forehead_path, axis=1)

In [95]:
new_val_df[new_val_df['forehead_path'].str.endswith('.JPG')]

,id,left_cheek_path,right_cheek_path,label,origin_forehead_path,forehead_path


In [96]:
import shutil

root_dir = '/home/work/hocheol_dir/workspace/data/034_cropped_data/0512_034_dataset'

for row in new_test_df.iterrows():
    shutil.copy(row[1]['origin_forehead_path'], os.path.join(root_dir, row[1]['forehead_path']))

for row in new_tr_df.iterrows():
    shutil.copy(row[1]['origin_forehead_path'], os.path.join(root_dir, row[1]['forehead_path']))

for row in new_val_df.iterrows():
    shutil.copy(row[1]['origin_forehead_path'], os.path.join(root_dir, row[1]['forehead_path']))

In [98]:
new_test_df.drop(columns=['origin_forehead_path'], inplace=True)
new_tr_df.drop(columns=['origin_forehead_path'], inplace=True)
new_val_df.drop(columns=['origin_forehead_path'], inplace=True)

In [130]:
origin_label = []
root_dirs = ['/home/work/hocheol_dir/workspace/data/034_cropped_data/034_forehead/034.마스크_라벨/034.마스크 착용 한국인 안면 이미지 데이터/01.데이터/1.Training/라벨링데이터_230522_add/output_dir/메타정보', 
                '/home/work/hocheol_dir/workspace/data/034_cropped_data/034_forehead/034.마스크_라벨/034.마스크 착용 한국인 안면 이미지 데이터/01.데이터/2.Validation/라벨링데이터_230524_add/메타정보']
for root_dir in root_dirs:
    for root, dirs, files in tqdm(os.walk(root_dir)):
        dirs[:] = [d for d in dirs if len(d) == 4]
        for file in files:
            if file.lower().endswith('.json'):
                if '19_01' in file or '46_01' in file:
                    with open(os.path.join(root, file), 'r') as f:
                        data = json.load(f)
                        data['id'] = None
                        data['path'] = os.path.join(root, file)
                        if '19_01' in file:
                            data['id'] = f"{file.split('_')[0]}_19"
                        elif '46_01' in file:
                            data['id'] = f"{file.split('_')[0]}_46"
                        origin_label.append(data)

2401it [00:03, 659.80it/s]
301it [00:00, 665.51it/s]


In [132]:
origin_label[0]

{'groundwork': 'CRS',
 'gender': 'M',
 'age': 28,
 'camera_type': 'SP',
 'camera_up_down_degree': '0',
 'camera_left_right_degree': '0',
 'distance_type': '0000',
 'mask_use_type': '01',
 'id': '0000_19',
 'path': '/home/work/hocheol_dir/workspace/data/034_cropped_data/034_forehead/034.마스크_라벨/034.마스크 착용 한국인 안면 이미지 데이터/01.데이터/1.Training/라벨링데이터_230522_add/output_dir/메타정보/0000/0000_CRS_19_01_meta.json'}

In [135]:
origin_label_df = pd.DataFrame(origin_label, columns=['id', 'age'])
origin_label_df

,id,age
0,0000_19,28.0
1,0000_46,28.0
2,0001_19,28.0
3,0001_46,28.0
4,0002_19,34.0
...,...,...
5395,2697_46,22.0
5396,2698_19,26.0
5397,2698_46,26.0
5398,2699_19,23.0


In [137]:
merged_test_df = pd.merge(new_test_df, origin_label_df, on='id', how='left')
merged_tr_df = pd.merge(new_tr_df, origin_label_df, on='id', how='left')
merged_val_df = pd.merge(new_val_df, origin_label_df, on='id', how='left')

In [141]:
len(merged_test_df), len(merged_tr_df), len(merged_val_df)

(532, 4245, 532)

In [ ]:
merged_test_df.sort_values(by='id', inplace=True)
merged_tr_df.sort_values(by='id', inplace=True)
merged_val_df.sort_values(by='id', inplace=True)

In [ ]:
with open('/home/work/hocheol_dir/workspace/data/034_cropped_data/0512_034_dataset/with_forehead/test.json', 'w') as f:
    json.dump(merged_test_df.to_dict(orient='records'), f, indent=2, ensure_ascii=False)

with open('/home/work/hocheol_dir/workspace/data/034_cropped_data/0512_034_dataset/with_forehead/train.json', 'w') as f:
    json.dump(merged_tr_df.to_dict(orient='records'), f, indent=2, ensure_ascii=False)

with open('/home/work/hocheol_dir/workspace/data/034_cropped_data/0512_034_dataset/with_forehead/val.json', 'w') as f:
    json.dump(merged_val_df.to_dict(orient='records'), f, indent=2, ensure_ascii=False)